# RAG Lab 3: End-to-End RAG Pipeline

### الهدف: بناء نظام توليد معزز بالاسترجاع (RAG) متكامل يدمج محرك البحث الدلالي مع نموذج التوليد GPT-2 للإجابة على الأسئلة بدقة بناءً على مستندات معرفية.


### أولاً: استدعاء نماذج التوليد والبحث وتحضير قاعدة المعرفة (Setup models & Database)

نقوم بتحميل نموذج المتجهات ونموذج التوليد GPT-2 وتجهيز قاعدة بيانات المعرفة التي سيبحث بها النظام.


In [ ]:
# Step 1) استدعاء النماذج وتجهيز قاعدة المعرفة / Load models and setup knowledge base
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
generator = pipeline("text-generation", model="gpt2")

knowledge_base = [
    "The capital of France is Paris. It is known for Eiffel Tower.",
    "The capital of Japan is Tokyo. It is famous for its sushi and technology.",
    "The capital of Australia is Canberra. It was selected as a compromise between Sydney and Melbourne."
]
kb_embeddings = embed_model.encode(knowledge_base)


### ثانياً: بناء دالة الاسترجاع الدلالي (Define retrieval function)

نقوم ببناء دالة تأخذ سؤال المستخدم وتبحث في قاعدة المعرفة لترجع المقطع الأكثر صلة وإفادة للاستعلام.


In [ ]:
# Step 2) بناء دالة الاسترجاع الدلالي / Define retrieval function
def retrieve(query):
    q_emb = embed_model.encode(query)
    scores = [np.dot(q_emb, kb_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(kb_emb)) for kb_emb in kb_embeddings]
    best_idx = np.argmax(scores)
    return knowledge_base[best_idx]


### ثالثاً: دمج السياق وصياغة الموجه وتوليد الإجابة (RAG execution)

نقوم بدمج السياق المجلوب مع سؤال المستخدم في قالب موجه (Prompt) موحد، ونرسله لنموذج التوليد لصياغة الإجابة الموثقة بالسياق.


In [ ]:
# Step 3) دمج السياق وتوليد الإجابة / RAG Execution
query = "What is the capital of Japan and what is it famous for?"
context = retrieve(query)

# Construct Prompt
prompt = f"Answer the query based on the context.\nContext: {context}\nQuery: {query}\nAnswer:"

# Generate text
output = generator(prompt, max_new_tokens=20, pad_token_id=50256)
print("RAG Prompt:")
print(prompt)
print("\nGenerated Response:")
print(output[0]['generated_text'])
